# Exploratory Data Analysis

This notebook performs a thorough exploratory data analysis (EDA) on the raw dataset. By extracting and visualizing metadata, you can:
- Understand the class balance and label distribution
- Inspect image dimensions, sizes, and aspect ratios
- Spot quality issues or duplicates
- Build intuition for preprocessing and modeling steps

## Imports and Configurations

We import all necessary Python libraries for data manipulation, visualization, and filesystem traversal.
- `os` for file and directory operations
- `pandas` for tabular data analysis
- `matplotlib` and `seaborn` for visualization
- `PIL.Image` for reading images and extracting dimensions
- `tqdm` for progress bars

We also define the base directory for the raw dataset and the expected split folders (`train` and `test`).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

# Base path to raw images
DATASET_DIR = 'data/raw'
splits = ['train', 'test']


## Metadata Collection

For every image in each split (`train`, `test`) and each label, we:
- Traverse the directory tree to find images
- Attempt to open and read the image with PIL (skipping unreadable/corrupt files)
- Extract metadata: label, split, filename, width, height, aspect ratio, file size (KB), and the full path

This generates a DataFrame with one row per image and useful metadata columns for further analysis.

In [ ]:
records = []
for split in splits:
    for label in os.listdir(os.path.join(DATASET_DIR, split)):
        folder = os.path.join(DATASET_DIR, split, label)
        for file in os.listdir(folder):
            path = os.path.join(folder, file)
            try:
                with Image.open(path) as img:
                    img = img.convert('RGB')
                    records.append({
                        'split': split,
                        'label': label,
                        'filename': file,
                        'width': img.width,
                        'height': img.height,
                        'aspect_ratio': round(img.width / img.height, 2),
                        'size_kb': os.path.getsize(path) / 1024,
                        'path': path
                    })
            except (UnidentifiedImageError, OSError):
                continue

df = pd.DataFrame(records)
df.head()


## Class Distribution

Visualize how samples are distributed across classes (`label`) and splits (`train`/`test`). This is important to check for class imbalance or split leakage.

In [ ]:
sns.countplot(data=df, x='label', hue='split')
plt.title("Class Distribution by Split")
plt.ylabel("Number of Images")
plt.show()


## Statistics

Show descriptive statistics for image width, height, aspect ratio, and file size. This helps:
- Spot outliers or corrupted images
- Guide preprocessing choices (e.g., resize dimensions, normalization)

In [ ]:
print(df[['width', 'height', 'aspect_ratio', 'size_kb']].describe())


### Image Size by Class

Boxplot of file size by class can reveal differences in data distribution, compression, or image sources.

In [ ]:
sns.boxplot(data=df, x='label', y='size_kb')
plt.title("Image Size (KB) by Class")
plt.show()


### Image Width vs Height

Scatter plot of width vs. height for each label. Helps spot outliers and understand the range of image dimensions.

In [ ]:
sns.scatterplot(data=df, x='width', y='height', hue='label', alpha=0.5)
plt.title("Image Width vs Height")
plt.show()


### Aspect Ratio Distribution

Histogram of image aspect ratios (width/height), grouped by class. Useful for:
- Detecting non-standard or inconsistent images
- Informing preprocessing (e.g., cropping vs. padding)

In [ ]:
sns.histplot(data=df, x='aspect_ratio', hue='label', bins=20, kde=True)
plt.title("Aspect Ratio Distribution by Class")
plt.show()


## (Optional) Duplicate Image Check

Commented-out code for detecting duplicate images using file hashes (MD5). Uncomment if you want to identify and investigate duplicates in your dataset.

In [ ]:
# def get_md5(path):
#     with open(path, 'rb') as f:
#         return hashlib.md5(f.read()).hexdigest()

# df['md5'] = df['path'].apply(get_md5)
# duplicates = df[df.duplicated('md5')]
# print("Duplicate images found:", len(duplicates))


## (Optional) Save Metadata and Summary

Commented-out code for saving the metadata DataFrame and a summary table to CSV files for future reference or sharing.

In [ ]:
# os.makedirs("data/output", exist_ok=True)
# df.to_csv("data/output/01_image_metadata.csv", index=False)
# df.groupby(['split', 'label']).size().to_csv("data/output/02_dataset_summary.csv")


## Display Sample Images

Display a couple of images from each class in the training set for visual inspection. This helps quickly spot any glaring issues with the data.

In [ ]:
from IPython.display import display
from PIL import Image

samples = df[df['split'] == 'train'].groupby('label').head(2)
for _, row in samples.iterrows():
    img = Image.open(row['path'])
    print(f"{row['label'].upper()} — {os.path.basename(row['path'])}")
    display(img.resize((150, 150)))
